# matmul-2d — worked example 2: Compute a Gram Matrix via Self-Matmul

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matmul-2d`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The Gram matrix of a data matrix `X` of shape `(N, D)` is `G = X @ X.T`, which has shape `(N, N)`. Entry `G[i, j]` equals the dot product between samples `i` and `j`. Gram matrices appear in kernel methods, style transfer losses, and covariance estimation. The contraction rule is `(N, D) @ (D, N) → (N, N)` — the inner dimension `D` is contracted away.

## Worked solution

**Step 1 — create a data matrix X of shape (5, 3).**
Five samples, each with 3 features. We use a fixed seed for reproducibility.

**Step 2 — transpose X to get X.T of shape (3, 5).**
The `.T` attribute swaps all dimensions of a 2-D tensor.

**Step 3 — compute the Gram matrix G = X @ X.T.**
The contraction `(5, 3) @ (3, 5) → (5, 5)` sums over the feature dimension. Each entry `G[i, j]` is the inner product of rows `i` and `j` of `X`.

**Step 4 — verify symmetry.**
A Gram matrix is always symmetric: `G == G.T`. We also verify the diagonal entries equal the squared norms of each row.

In [ ]:
import torch as t

t.manual_seed(9)
X = t.randn(5, 3)  # 5 samples, 3 features

# Gram matrix: (5,3) @ (3,5) -> (5,5)
G = X @ X.T

print(f"X shape:  {X.shape}")
print(f"X.T shape: {X.T.shape}")
print(f"G shape:   {G.shape}")
print(f"G symmetric: {t.allclose(G, G.T)}")

# Diagonal entries should equal squared row norms
row_norms_sq = (X ** 2).sum(dim=1)   # shape (5,)
print(f"\nDiag(G):       {G.diagonal().round(decimals=3)}")
print(f"||row||^2:     {row_norms_sq.round(decimals=3)}")
print(f"Diag match:    {t.allclose(G.diagonal(), row_norms_sq)}")

# Also check one off-diagonal entry manually
i, j = 0, 2
dot_ij = (X[i] * X[j]).sum()
print(f"\nG[0,2]={G[0,2]:.4f}  manual dot={dot_ij:.4f}  match={t.isclose(G[0,2], dot_ij)}")